# Problem Statement

Design a time-based key-value data structure that can store multiple values for the same key at different timestamps and retrieve the key's value at a certain timestamp.

Implement the TimeMap class:

    TimeMap() Initializes the object.

    void set(String key, String value, int timestamp) Stores the key key with the value value at the given time timestamp.

    String get(String key, int timestamp) Returns a value such that set was called previously, with timestamp_prev <= timestamp. If there are multiple such values, return the value associated with the largest timestamp_prev. If there are no values, return "".

Note: All timestamps passed to set are strictly increasing.

## Example 1:


Input:
["TimeMap", "set", "get", "get", "set", "get", "get"]
[[], ["foo", "bar", 1], ["foo", 1], ["foo", 3], ["foo", "bar2", 4], ["foo", 4], ["foo", 5]]

Output:
[null, null, "bar", "bar", null, "bar2", "bar2"]

Explanation:
TimeMap timeMap = new TimeMap();
timeMap.set("foo", "bar", 1);  // store key "foo" and value "bar" at timestamp = 1
timeMap.get("foo", 1);         // return "bar"
timeMap.get("foo", 3);         // return "bar", no value at timestamp 3, so use timestamp 1
timeMap.set("foo", "bar2", 4); // store key "foo" and value "bar2" at timestamp = 4
timeMap.get("foo", 4);         // return "bar2"
timeMap.get("foo", 5);         // return "bar2"

## Constraints:

    1 <= key.length, value.length <= 100

    key and value consist of lowercase English letters and digits

    1 <= timestamp <= 10^7

    All timestamps of set are strictly increasing

    At most 2 * 10^5 calls will be made to set and get

In [1]:
class TimeMap:
    def __init__(self):
        self.store = {}

    def set(self, key: str, value: str, timestamp: int) -> None:
        if key not in self.store:
            self.store[key] = []
        self.store[key].append((timestamp, value))

    def get(self, key: str, timestamp: int) -> str:
        if key not in self.store:
            return ""

        pairs = self.store[key]

        # Early returns for edge cases
        if not pairs:
            return ""
        if pairs[0][0] > timestamp:
            return ""
        if pairs[-1][0] <= timestamp:
            return pairs[-1][1]

        # Binary search
        left, right = 0, len(pairs) - 1
        result = ""

        while left <= right:
            mid = (left + right) // 2
            if pairs[mid][0] <= timestamp:
                result = pairs[mid][1]
                left = mid + 1  # Continue searching right for larger timestamp
            else:
                right = mid - 1

        return result

In [2]:
def test_time_map():
    # Test 1: Basic operations
    timeMap = TimeMap()
    timeMap.set("foo", "bar", 1)
    assert timeMap.get("foo", 1) == "bar"
    assert timeMap.get("foo", 3) == "bar"  # Should return bar (timestamp 1)

    timeMap.set("foo", "bar2", 4)
    assert timeMap.get("foo", 4) == "bar2"
    assert timeMap.get("foo", 5) == "bar2"

    # Test 2: Non-existent key
    assert timeMap.get("baz", 1) == ""

    # Test 3: Multiple keys
    timeMap.set("apple", "red", 2)
    timeMap.set("apple", "green", 5)
    timeMap.set("apple", "yellow", 8)

    assert timeMap.get("apple", 1) == ""
    assert timeMap.get("apple", 2) == "red"
    assert timeMap.get("apple", 4) == "red"  # Closest is timestamp 2
    assert timeMap.get("apple", 5) == "green"
    assert timeMap.get("apple", 7) == "green"  # Closest is timestamp 5
    assert timeMap.get("apple", 8) == "yellow"
    assert timeMap.get("apple", 10) == "yellow"

    # Test 4: Timestamp before first entry
    timeMap.set("test", "value", 10)
    assert timeMap.get("test", 5) == ""

    # Test 5: Many values for same key
    for i in range(1000):
        timeMap.set("many", f"value{i}", i)

    assert timeMap.get("many", 500) == "value500"
    assert timeMap.get("many", 999) == "value999"
    assert timeMap.get("many", 1000) == "value999"  # Max is 999

    print("All tests passed! ✓")


test_time_map()

All tests passed! ✓
